# Aula 4 — Sinal v1: Momentum Cross-Seccional

**Intensivão Quant AI — ImpactUFSCar**

---

Esta é a aula central do intensivão. Você vai construir o sinal que vai gerar todos os sinais de compra e venda da estratégia. Tudo que vem antes foi preparação; tudo que vem depois é refinamento e validação.

Ao final, você terá:
- Entendido por que momentum funciona através de probabilidade condicional
- Calculado o sinal 12-1 para todas as ações do IBOVESPA
- Montado a primeira carteira com equal weight

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.
# Qualquer pessoa que clonar o repositório pode rodar sem modificações.

import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow',
                    'statsmodels', 'python-docx', 'anthropic'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ], check=False)
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Garante que pyarrow está instalado (necessário para ler/salvar parquet)
    try:
        import pyarrow
    except ImportError:
        print("Instalando pyarrow...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'],
                       check=False)

    # Localiza a raiz do repositório subindo a árvore de diretórios até o .git
    # Funciona independente de onde o usuário clonou o projeto
    _p = os.path.abspath(os.getcwd())
    _root = None
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            _root = _p
            break
        _p = os.path.dirname(_p)

    # Fallback: se não encontrar .git, usa a pasta pai do notebook
    if _root is None:
        _root = os.path.dirname(os.path.abspath('__file__'))
        print("  Aviso: repositório .git não encontrado. Usando pasta do notebook.")

    DADOS_DIR = os.path.join(_root, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados em: {DADOS_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Dados limpos da Aula 3
ret_mensais = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
tickers     = pd.read_csv(os.path.join(DADOS_DIR, 'tickers_finais.csv'))['ticker'].tolist()

print(f"Dataset: {ret_mensais.shape[0]} meses × {ret_mensais.shape[1]} ações")
print(f"Período: {ret_mensais.index[0].date()} a {ret_mensais.index[-1].date()}")

In [ ]:
# ── VERIFICAÇÃO DE DEPENDÊNCIAS ──────────────────────────────────────────────
# Este notebook depende dos dados gerados pela aula anterior.
# Se algum arquivo estiver faltando, rode o notebook indicado primeiro.

_arquivos_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'),
]

_faltando = [f for f in _arquivos_necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  ATENÇÃO: arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   Faltando: {os.path.basename(f)}")
    print(f"\n   Execute primeiro o notebook: aula-03-eda")
    print(f"   Os dados devem ficar em: {DADOS_DIR}")
else:
    print("✓  Todos os arquivos necessários encontrados.")

## 1. Por que o Momentum Funciona? A Visão da Finança Comportamental

A estratégia de **Momentum** (também conhecida na literatura como *Cross-Sectional Momentum*) baseia-se em uma premissa extremamente simples e intuitiva: **ativos que performaram muito bem no passado recente tendem a continuar performando bem no futuro próximo, enquanto ativos perdedores tendem a continuar performando mal.**

Essa anomalia de mercado foi amplamente documentada por **Jegadeesh e Titman (1993)** no mercado norte-americano, e posteriormente validada em praticamente todos os mercados acionários do mundo (incluindo o Brasil).

---

### As Explicações Comportamentais (Por que o mercado é ineficiente?)

A teoria de finanças clássica afirma que o momentum não deveria existir, pois novas informações deveriam ser incorporadas aos preços instantaneamente (Hipótese de Mercado Eficiente). No entanto, o mercado é feito de seres humanos propensos a vieses cognitivos. São três os principais fatores comportamentais que explicam por que o momentum funciona:

#### 1. Sub-reação Inicial (Underreaction)
Quando uma empresa divulga resultados surpreendentemente bons (notícia altamente positiva), os investidores não ajustam o preço da ação para o valor justo de forma imediata. Isso ocorre por dois vieses:
- **Ancoragem:** Os investidores se apegam a referências de preços passados e relutam em aceitar que a empresa vale muito mais agora.
- **Viés de Conservadorismo:** Os analistas de mercado tendem a atualizar suas estimativas de lucros de forma muito lenta e gradual, fazendo com que o preço suba lentamente em direção ao valor justo em vez de saltar de uma vez.

#### 2. Difusão Lenta da Informação
A informação relevante não chega a todos os participantes do mercado ao mesmo tempo. Primeiro, os investidores institucionais compram, depois os analistas divulgam relatórios, depois a notícia chega à mídia de varejo. Essa propagação gradual cria uma tendência de alta contínua ao longo de meses.

#### 3. Sob-reação Posterior (Overreaction / Efeito Manada)
À medida que uma ação sobe consecutivamente por vários meses, ela começa a atrair a atenção de investidores que não querem ficar de fora da festa (fenômeno conhecido como **FOMO** — *Fear Of Missing Out*).
- O **efeito manada (herding behavior)** faz com que mais investidores comprem simplesmente porque o preço está subindo, criando um ciclo de feedback positivo que empurra o preço ainda mais para cima, extrapolando o valor fundamental do ativo temporariamente.


In [ ]:
# Teste empírico de probabilidade condicional
# Pergunta: ações no quartil superior do retorno passado (3 meses)
#           têm retorno futuro (1 mês) maior que as do quartil inferior?

# Retorno dos últimos 3 meses (sinal simples)
ret_passado = ret_mensais.shift(1).rolling(3).sum()

# Retorno do próximo mês (o que queremos prever)
ret_futuro = ret_mensais

# Para cada mês, classifica as ações em quartis pelo retorno passado
resultados = []
for data in ret_passado.index:
    passado = ret_passado.loc[data].dropna()
    futuro  = ret_futuro.loc[data].dropna() if data in ret_futuro.index else pd.Series()

    ativos_comuns = passado.index.intersection(futuro.index)
    if len(ativos_comuns) < 10:
        continue

    p = passado[ativos_comuns]
    f = futuro[ativos_comuns]

    quartis = pd.qcut(p, 4, labels=['Q1 (piores)', 'Q2', 'Q3', 'Q4 (melhores)'])
    for q in ['Q1 (piores)', 'Q4 (melhores)']:
        mask = quartis == q
        if mask.sum() > 0:
            resultados.append({'data': data, 'quartil': q, 'retorno_futuro': f[mask].mean()})

df_resultados = pd.DataFrame(resultados)

# Média por quartil
media_por_quartil = df_resultados.groupby('quartil')['retorno_futuro'].mean()
print("Retorno médio no mês seguinte por quartil de momentum passado (3m):")
for q, v in media_por_quartil.items():
    print(f"  {q}: {v:.2%} ao mês")
print()
print(f"Diferença Q4 − Q1: {media_por_quartil['Q4 (melhores)'] - media_por_quartil['Q1 (piores)']:.2%} ao mês")

In [ ]:
# Visualização: distribuição dos retornos futuros por quartil
fig, ax = plt.subplots(figsize=(11, 5))

for quartil, grupo in df_resultados.groupby('quartil'):
    grupo['retorno_futuro'].hist(
        ax=ax, bins=40, alpha=0.5, density=True, label=quartil
    )

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Distribuição do retorno futuro por quartil de momentum passado')
ax.set_xlabel('Retorno no mês seguinte')
ax.set_ylabel('Densidade')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Cross-Sectional Momentum vs. Time-Series Momentum

É muito comum iniciantes confundirem os dois principais tipos de Momentum. Embora ambos explorem tendências, suas formulações matemáticas e conceitos de risco são completamente distintos.

---

### A. Cross-Sectional Momentum (Momentum Cross-Seccional)
- **Como funciona:** É um conceito **relativo**. Nós olhamos para um universo fechado de ativos (por exemplo, todas as componentes do IBOVESPA) e comparamos a performance delas entre si.
- **A Operação:** Nós rankeamos as ações do melhor para o pior retorno no período de sinal. Em seguida, compramos o topo do ranking (vencedores) e, se a estratégia for long-short, vendemos a base do ranking (perdedores).
- **Características:** A carteira sempre mantém exposição ativa líquida controlada. Se o mercado todo estiver em queda livre, a estratégia selecionará como vencedoras as ações que caíram menos no período.

---

### B. Time-Series Momentum (Momentum Temporal / Trend Following)
- **Como funciona:** É um conceito **absoluto**. Cada ativo é julgado individualmente em relação ao seu próprio histórico de preços, sem comparação com os outros.
- **A Operação:** Se uma ação específica tem retorno acumulado positivo (ou está acima de uma média móvel de longo prazo), nós compramos. Se estiver com tendência de queda absoluta, nós vendemos ou ficamos de fora.
- **Características:** A exposição líquida da carteira varia drasticamente. Em mercados de baixa generalizada, a estratégia pode ficar $100\%$ alocada em caixa, protegendo o capital.


## 3. A Janela de Momentum 12-1 e a Justificativa da Reversão de Curto Prazo

Na literatura acadêmica e na prática de gestoras quantitativas, a janela padrão clássica para o cálculo de momentum de médio prazo é a janela **12-1 meses** (12 a 1).

### O que significa "12-1"?
Significa calcular o retorno acumulado de um ativo nos últimos 12 meses, mas **excluindo/pulando o mês mais recente (mês $t-1$)**.

```
Passado                                                  Presente (Hoje)
[ Mês -12 | Mês -11 | ... | Mês -3 | Mês -2 ] | [ Mês -1 (PULADO) ] | [ Instante de Decisão ]
<------------- Janela de Cálculo ------------->
```

---

### Por que pulamos o mês mais recente? (Justificativa Microestrutural)

Se incluirmos o retorno do último mês na nossa métrica de momentum de médio prazo, o desempenho da nossa estratégia cai significativamente. Isso ocorre devido ao fenômeno da **Reversão de Curto Prazo (Short-Term Reversal)**.

Existem duas razões principais para essa reversão de curtíssimo prazo:

#### 1. Pressão Temporária de Liquidez e Inventário de Market Makers
Se grandes fundos precisam liquidar rapidamente posições em um ativo no final do mês devido a resgates, eles geram uma pressão vendedora artificial violenta, derrubando o preço temporariamente abaixo do seu valor justo. No mês seguinte, esse desequilíbrio é corrigido por investidores provendo liquidez, fazendo com que o preço retorne à sua média rapidamente.

#### 2. Efeito Bid-Ask Bounce (Salto do Spread)
Em ações com spreads mais largos entre compra e venda, o fechamento diário pode saltar artificialmente dependendo se a última operação foi executada na ponta de compra (ask) ou de venda (bid). Esse ruído estatístico gera uma autocorrelação negativa artificial de curtíssimo prazo.

> **Conclusão de Ouro:** O Momentum real de tendência econômica é um fenômeno de **médio prazo** (de 3 a 12 meses). O curtíssimo prazo (1 mês ou menos) é dominado por ruído microestrutural e reversão à média. Por isso, a regra industrial quant é **pular o mês mais recente** (usar a janela 12-1 ou similar) para capturar a tendência limpa de médio prazo.


In [ ]:
# Calculando o sinal 12-1
# Em cada mês M, o sinal usa retornos de M-12 até M-2 (11 meses, skip de 1)
# shift(2): o retorno mais recente na janela foi 2 meses atrás
# rolling(11): soma 11 meses consecutivos

sinal = ret_mensais.shift(2).rolling(11).sum()

print("Sinal calculado.")
print(f"Shape: {sinal.shape}")
print(f"Primeiros dados disponíveis: {sinal.dropna(how='all').index[0].date()}")
print(f"  (13 meses após o início dos retornos — esperado)")
print()
print("Sinal do último mês disponível — top 5:")
ultimo_mes = sinal.iloc[-1].dropna().sort_values(ascending=False)
print(ultimo_mes.head().rename(lambda x: x.replace('.SA','')))

In [ ]:
# Visualizando o sinal ao longo do tempo para algumas ações
acoes_exemplo = ['PETR4.SA', 'VALE3.SA', 'WEGE3.SA', 'MGLU3.SA']
acoes_plot    = [a for a in acoes_exemplo if a in sinal.columns]

fig, ax = plt.subplots(figsize=(13, 5))
for acao in acoes_plot:
    sinal[acao].plot(ax=ax, label=acao.replace('.SA', ''), alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Sinal de momentum 12-1 ao longo do tempo')
ax.set_ylabel('Retorno acumulado (11 meses, t-2)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. O Processo de Ranking Cross-Seccional

Após calcularmos a métrica do sinal (retorno acumulado 12-1) para cada ativo do universo, nós não usamos esses valores percentuais diretamente na alocação. Em vez disso, nós aplicamos uma transformação de **Ranking**.

---

### Por que transformar retornos em postos/ranks?

1. **Robustez contra Outliers:** Se uma única ação específica subiu $500\%$ em 11 meses devido a uma especulação atípica de fusão, o sinal bruto dessa ação seria gigantesco e distorceria qualquer peso ponderado. Ao transformar em ranking (onde ela será simplesmente a número 1 daquele mês), limitamos o impacto de retornos extremos no portfólio.
2. **Escala invariante:** Os rankings convertem valores brutos heterogêneos de retornos mensais em percentis distribuídos uniformemente entre 0 e 1.
3. **Spearman Information Coefficient (IC):** O poder preditivo do sinal é medido via correlação de postos (Rank Correlation) de Spearman entre o ranking do sinal e o ranking dos retornos futuros reais do ativo. Isso nos diz, de forma honesta, se nossa ordenação de "melhores para piores" no início do mês realmente se traduziu em maiores retornos no final do mês.


In [ ]:
# Rank percentual — calculado linha a linha (cada mês é uma linha)
ranking = sinal.rank(axis=1, ascending=True, pct=True)

print("Ranking percentual — último mês disponível (top 5 e bottom 5):")
ult_ranking = ranking.iloc[-1].dropna().sort_values(ascending=False)

print("\nTop 5 (maiores ranks):")
print(ult_ranking.head().rename(lambda x: x.replace('.SA','')).round(3))

print("\nBottom 5 (menores ranks):")
print(ult_ranking.tail().rename(lambda x: x.replace('.SA','')).round(3))

## 5. Long-Only vs. Long-Short: Estruturas de Portfólio

Ao construir um portfólio quantitativo, o gestor precisa decidir como expressar o sinal do modelo no mercado de acordo com as restrições regulatórias e o perfil de risco do fundo.

---

### A. Long-Only (Apenas Comprado)
- **O que é:** O fundo compra apenas ativos cujos sinais são positivos (os vencedores). A exposição comprada total da carteira soma $100\%$ do patrimônio líquido.
- **Vantagens:** Simples de gerenciar, não possui custos de aluguel de ações, tem risco de perda limitado ao capital investido e é o formato padrão da imensa maioria dos fundos mútuos de ações abertos para investidores em geral no Brasil.
- **Desvantagens:** Totalmente exposto a movimentos de queda do mercado amplo (Beta do mercado próximo a 1). Se o IBOVESPA cair $30\%$, uma carteira momentum long-only provavelmente também cairá significativamente.

---

### B. Long-Short (Comprado e Vendido)
- **O que é:** O fundo compra os melhores ativos do ranking (long) e simultaneamente vende a descoberto (*short*) os piores ativos do ranking. A exposição líquida costuma ser de mercado neutro (*market-neutral*), ex: $50\%$ comprado e $50\%$ vendido.
- **Vantagens:** O rendimento da carteira depende exclusivamente da habilidade do modelo em ordenar os ativos. Se o mercado desabar $30\%$, o ganho nas posições vendidas compensa a perda das compradas.
- **Desvantagens:** Extremamente complexo operativamente. Requer aluguel de ações, gera taxas de corretagem em dobro, está sujeito a chamadas de margem diárias e possui risco de perdas ilimitadas nas pontas vendidas se as ações dispararem de preço repentinamente.


## 6. Montando a carteira — Equal Weight

Selecionamos as **top N ações** em cada mês e damos peso igual a cada uma: cada ação recebe $1/N$ do portfólio.

Vantagens do equal weight:
- Simples e transparente
- Não depende de estimativas de covariância (que são ruidosas)
- Diversificação máxima dentro do universo selecionado

Na Aula 6 vamos comparar com signal-weighted e outras abordagens.

In [ ]:
def calcular_pesos_equal_weight(ranking_row, N=10):
    """Seleciona top N ações e dá peso igual a cada uma."""
    validos = ranking_row.dropna()
    if len(validos) < N:
        return pd.Series(0.0, index=ranking_row.index)

    threshold = validos.nlargest(N).min()  # menor rank entre os top N
    selecionados = validos[validos >= threshold].index[:N]  # garante exatamente N

    pesos = pd.Series(0.0, index=ranking_row.index)
    pesos[selecionados] = 1.0 / N
    return pesos

N_ATIVOS = 10
pesos = ranking.apply(calcular_pesos_equal_weight, axis=1, N=N_ATIVOS)

# Verificação: pesos somam 1 (ou 0 nos meses sem sinal)
soma_pesos = pesos.sum(axis=1)
meses_com_carteira = (soma_pesos > 0).sum()
print(f"Meses com carteira formada: {meses_com_carteira} de {len(pesos)}")
print(f"Soma dos pesos (deve ser 1.0): {soma_pesos[soma_pesos > 0].mean():.4f}")

In [ ]:
# Quais ações aparecem mais frequentemente no top 10?
frequencia = (pesos > 0).sum().sort_values(ascending=False)
total_meses = meses_com_carteira

fig, ax = plt.subplots(figsize=(13, 4))
(frequencia.head(20) / total_meses * 100).plot(
    kind='bar', ax=ax, color='steelblue', width=0.8
)
ax.set_title(f'Frequência no top {N_ATIVOS} — % dos meses em que a ação foi selecionada')
ax.set_ylabel('% dos meses')
ax.set_xticklabels(
    [t.replace('.SA', '') for t in frequencia.head(20).index], rotation=45
)
plt.tight_layout()
plt.show()

In [ ]:
# Composição da carteira no último mês
ult_pesos = pesos.iloc[-1]
carteira_atual = ult_pesos[ult_pesos > 0].rename(lambda x: x.replace('.SA',''))

print(f"Carteira momentum — {pesos.index[-1].strftime('%b/%Y')}:")
for acao, peso in carteira_atual.items():
    print(f"  {acao:<12} {peso:.1%}")

In [ ]:
# Turnover mensal — quantas ações trocam a cada mês?
# Métrica importante: alto turnover = altos custos de transação (veremos na Aula 8)

carteira_binaria = (pesos > 0).astype(int)
entradas = (carteira_binaria.diff() > 0).sum(axis=1)   # ações que entraram
saidas   = (carteira_binaria.diff() < 0).sum(axis=1)   # ações que saíram
turnover = ((entradas + saidas) / 2) / N_ATIVOS          # % da carteira renovada

print(f"Turnover médio mensal: {turnover.mean():.1%} da carteira")
print(f"  → {turnover.mean() * N_ATIVOS:.1f} ações trocam por mês em média (de {N_ATIVOS})")

fig, ax = plt.subplots(figsize=(13, 3))
turnover.plot(ax=ax, color='steelblue', alpha=0.8)
ax.axhline(turnover.mean(), color='red', linestyle='--',
           label=f'média: {turnover.mean():.1%}')
ax.set_title('Turnover mensal da carteira')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 7. Retorno da carteira

Com os pesos calculados, podemos calcular o retorno mensal da carteira:

$$r_{\text{carteira},t} = \sum_{i} w_{i,t} \cdot r_{i,t}$$

Este é o produto vetorial entre pesos e retornos — **álgebra linear que você vai expandir na Aula 6**. Por ora, calculamos e plotamos o resultado.

In [ ]:
# Retorno mensal da carteira = soma ponderada dos retornos
# pesos contém os pesos decididos NO INÍCIO do mês
# ret_mensais contém os retornos REALIZADOS no mês
# Alinhamento: os pesos de M são aplicados ao retorno de M

ret_carteira = (pesos * ret_mensais).sum(axis=1)

# Remove meses sem carteira formada
ret_carteira = ret_carteira[pesos.sum(axis=1) > 0]

# Retorno acumulado
acum_carteira = ret_carteira.cumsum()

# Benchmark: IBOVESPA
import yfinance as yf
ibov_raw = yf.download('^BVSP', start='2010-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)['Close'].squeeze()
ibov_ret  = np.log(ibov_raw / ibov_raw.shift(1))
ibov_mens = ibov_ret.resample('ME').sum()
ibov_acum = ibov_mens.reindex(acum_carteira.index).cumsum()

fig, ax = plt.subplots(figsize=(13, 5))
acum_carteira.plot(ax=ax, label=f'Momentum top {N_ATIVOS} (equal weight)', linewidth=2)
ibov_acum.plot(ax=ax, label='IBOVESPA', linewidth=1.5, linestyle='--', color='gray')
ax.set_title('Retorno acumulado — Momentum v1 vs IBOVESPA')
ax.set_ylabel('Retorno log acumulado')
ax.legend()
plt.tight_layout()
plt.show()

print("Preview das métricas (versão completa na Aula 5):")
vol_mens = ret_carteira.std()
sharpe_aprox = (ret_carteira.mean() / vol_mens) * np.sqrt(12)
print(f"  Retorno médio mensal: {ret_carteira.mean():.2%}")
print(f"  Sharpe anualizado (aprox): {sharpe_aprox:.2f}")

## 8. Salvando o sinal v1

In [ ]:
sinal.to_parquet(os.path.join(DADOS_DIR, 'sinal_v1.parquet'))
pesos.to_parquet(os.path.join(DADOS_DIR, 'pesos_v1.parquet'))
ret_carteira.to_frame('ret_momentum').to_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet'))

print("Arquivos salvos:")
print("  sinal_v1.parquet           — sinal 12-1 para todas as ações")
print("  pesos_v1.parquet           — pesos mensais (equal weight top 10)")
print("  retorno_carteira_v1.parquet — retorno mensal da carteira")

## Resumo da aula

| Conceito | Implementação |
|---|---|
| Probabilidade condicional | Q4 de momentum tem retorno futuro maior que Q1 — verificado empiricamente |
| Cross-seccional | Rankear ações entre si a cada mês |
| Janela 12-1 | `ret_mensais.shift(2).rolling(11).sum()` |
| Equal weight | Top 10, cada um com 10% da carteira |
| Turnover | ~30-50% ao mês — alto, vai custar na Aula 8 |

---

## Para replicar em casa

1. Rode o notebook do início ao fim
2. Experimente mudar `N_ATIVOS` para 5, 15, 20 — como muda o retorno acumulado?
3. Experimente mudar a janela do sinal: substitua `shift(2).rolling(11)` por `shift(2).rolling(5)` (6-1) ou `shift(2).rolling(8)` (9-1) — qual funciona melhor?

> **Atenção:** testar muitas combinações e ficar com a melhor é overfitting. Na Aula 8 você vai aprender a corrigir isso. Por ora, explore livremente — mas anote que está explorando.

---

*ImpactUFSCar — Diretoria de Quant*